In [1]:
RERUN = False          # DNA_RERUN=1 in the environment turns the GPU rerun cells on
from _common import *
from _dna_helpers import *

RERUN = rerun_flag(RERUN)
P = paper()
LENGTHS = [50, 100, 200, 400]

# 04 · Exactly computable DNA rewards (Table 13, App. F.1 text)

Paper §3.1 (last paragraph) and App. F.1. Rewards: a smooth maximum over occurrences of `TTTTTC` ("motif") and a
smooth minimum of the `TTTTTC` and `AAAATT` occurrence scores ("conjunction"), τ = 0.1. For each length, the
target score is the 90th percentile of 1,024 unconditional base-flow samples. 100 guided/unguided pairs per
setting with identical initial noise, one-step composed dMFM flow map as posterior sampler, mc = 8,
guidance_frac = 8, guidance window t ∈ [0.5, 0.95] — produced by `scripts/sample_parent_reward_attainment.py`
(Slurm jobs 35383601–35383680, one per reward × length) and scored with Evo 2-7B by
`scripts/score_parent_reward_attainment_evo2.py` (jobs 35387510–35387523).

**Status: reproduced exactly** from the stored per-pair scores (`exact_rewards/<reward>/L<L>/reward_attainment_samples.csv`).

Paired error reduction per pair = |s_unguided − s⋆| − |s_guided − s⋆|. The 95% CI is a percentile bootstrap
of the mean, 2,000 resamples with `numpy.random.default_rng(seed + L).integers(...)`, seed = 20260731, exactly as in the
original script.

In [2]:
rows, per_setting = [], []
for reward in ["motif", "conjunction"]:
    for L in LENGTHS:
        d = load_csv(f"dna/exact_rewards/{reward}/L{L}/reward_attainment_samples.csv")
        md = load_json(f"dna/exact_rewards/{reward}/L{L}/metadata.json")
        tgt = md["target_score"]
        delta = np.abs(d.score_unguided.to_numpy() - tgt) - np.abs(d.score_guided.to_numpy() - tgt)
        lo, hi = bootstrap_integers(delta, md["generation"]["bootstrap"], md["generation"]["seed"] + L)
        pv = P["table13_exact_rewards"][reward][str(L)]
        rows += [dict(item=f"{reward} L{L} paired reduction", paper=pv[0], ours=float(delta.mean()), dec=3),
                 dict(item=f"{reward} L{L} CI low", paper=pv[1], ours=lo, dec=3),
                 dict(item=f"{reward} L{L} CI high", paper=pv[2], ours=hi, dec=3)]
        per_setting.append(dict(reward=reward, L=L, n_pairs=len(d), target_score=tgt, frac_improved=float((delta > 0).mean()),
                                paired_reduction=float(delta.mean()), ci_low=lo, ci_high=hi))
t13 = show(cmp(rows))
t13

status counts: {'match': 24}


,item,paper,ours,status,tol,tol_src,note
0,motif L50 paired reduction,0.126,0.126334,match,0.0005,atol,
1,motif L50 CI low,0.109,0.109186,match,0.0005,atol,
2,motif L50 CI high,0.144,0.144182,match,0.0005,atol,
3,motif L100 paired reduction,0.106,0.106485,match,0.0005,atol,
4,motif L100 CI low,0.093,0.093242,match,0.0005,atol,
5,motif L100 CI high,0.121,0.121064,match,0.0005,atol,
6,motif L200 paired reduction,0.095,0.094716,match,0.0005,atol,
7,motif L200 CI low,0.084,0.084115,match,0.0005,atol,
8,motif L200 CI high,0.105,0.105038,match,0.0005,atol,
9,motif L400 paired reduction,0.066,0.065603,match,0.0005,atol,


In [3]:
settings = pd.DataFrame(per_setting)
settings

,reward,L,n_pairs,target_score,frac_improved,paired_reduction,ci_low,ci_high
0,motif,50,100,0.587551,0.98,0.126334,0.109186,0.144182
1,motif,100,100,0.601449,0.96,0.106485,0.093242,0.121064
2,motif,200,100,0.587543,0.98,0.094716,0.084115,0.105038
3,motif,400,100,0.585716,0.94,0.065603,0.056270,0.075418
4,conjunction,50,100,0.538077,0.95,0.092471,0.080032,0.105854
5,conjunction,100,100,0.550778,0.94,0.067056,0.058373,0.076205
6,conjunction,200,100,0.547419,0.97,0.057039,0.050077,0.064013
7,conjunction,400,100,0.558824,0.93,0.043990,0.037512,0.050520


## App. F.1 text claims

- "Guidance improves 93–98% of matched pairs across the eight settings" (§3.1: "around 95%").
- "Every generated sample is unique."
- "normalized pairwise Hamming diversity decreases by only 0.008–0.020" — recomputed here from the 100 sequences of
  each set (i < j pairs), and checked against the stored summary.
- "nearest-test distance decreases by 0.003–0.012" — needs the held-out test set (7k–8k windows), so it is read from
  the stored `reward_attainment_summary.csv` (computed by the original script against the full `test` split).
- "Evo 2 NLL improves in every setting" — means recomputed from `evo2/evo2_per_sequence.csv` (forward/reverse-complement
  averaged NLL per sequence).

In [4]:
claims = []
for reward in ["motif", "conjunction"]:
    for L in LENGTHS:
        d = load_csv(f"dna/exact_rewards/{reward}/L{L}/reward_attainment_samples.csv")
        s = load_csv(f"dna/exact_rewards/{reward}/L{L}/reward_attainment_summary.csv").set_index("set")
        e = load_csv(f"dna/exact_rewards/{reward}/L{L}/evo2/evo2_per_sequence.csv")
        pw = {k: float(pairwise_hamming_frac(tokens(d[f"sequence_{k}"])).mean()) for k in ["unguided", "guided"]}
        nll = e.groupby("set").evo2_nll.mean()
        claims.append(dict(reward=reward, L=L, n_pairs=len(d),
                           unique_unguided=d.sequence_unguided.nunique() / len(d), unique_guided=d.sequence_guided.nunique() / len(d),
                           pairwise_decrease=pw["unguided"] - pw["guided"],
                           pairwise_decrease_stored=float(s.loc["unguided", "pairwise_hamming_frac"] - s.loc["guided", "pairwise_hamming_frac"]),
                           nearest_test_decrease_stored=float(s.loc["unguided", "nearest_test_hamming_frac"] - s.loc["guided", "nearest_test_hamming_frac"]),
                           evo2_nll_unguided=float(nll["unguided"]), evo2_nll_guided=float(nll["guided"]), evo2_nll_heldout=float(nll["heldout_test"])))
claims = pd.DataFrame(claims)
claims["evo2_improves"] = claims.evo2_nll_guided < claims.evo2_nll_unguided
claims

,reward,L,n_pairs,unique_unguided,unique_guided,pairwise_decrease,pairwise_decrease_stored,nearest_test_decrease_stored,evo2_nll_unguided,evo2_nll_guided,evo2_nll_heldout,evo2_improves
0,motif,50,100,1.0,1.0,0.014505,0.014505,0.009400,1.359098,1.343611,1.324737,True
1,motif,100,100,1.0,1.0,0.015499,0.015499,0.007500,1.359830,1.343868,1.282074,True
2,motif,200,100,1.0,1.0,0.009571,0.009571,0.003150,1.362900,1.347048,1.269909,True
3,motif,400,100,1.0,1.0,0.007609,0.007609,0.004425,1.346498,1.329908,1.185547,True
4,conjunction,50,100,1.0,1.0,0.019677,0.019677,0.012400,1.359098,1.337228,1.324737,True
5,conjunction,100,100,1.0,1.0,0.017947,0.017947,0.009100,1.359830,1.333504,1.282074,True
6,conjunction,200,100,1.0,1.0,0.011960,0.011960,0.004900,1.362900,1.340131,1.269909,True
7,conjunction,400,100,1.0,1.0,0.010528,0.010528,0.004050,1.346498,1.325593,1.185547,True


In [5]:
pt = P["table13_exact_rewards"]
rows = [
    dict(item="min fraction of pairs improved", paper=pt["text_frac_pairs_improved_range"][0], ours=settings.frac_improved.min(), dec=2),
    dict(item="max fraction of pairs improved", paper=pt["text_frac_pairs_improved_range"][1], ours=settings.frac_improved.max(), dec=2),
    dict(item="mean fraction improved (§3.1 'around 95%')", paper=P["sec3_1_text"]["exact_rewards_frac_pairs_improved_approx"], ours=settings.frac_improved.mean(), atol=0.01, note="approximate claim ('around'): tolerance 0.01"),
    dict(item="pairs per setting (min over the 8)", paper=pt["text_n_pairs"], ours=float(claims.n_pairs.min()), atol=0),
    dict(item="pairs per setting (max over the 8)", paper=pt["text_n_pairs"], ours=float(claims.n_pairs.max()), atol=0),
    dict(item="all samples unique (1 = yes)", paper=1, ours=float((claims[["unique_unguided", "unique_guided"]] == 1).all().all()), atol=0),
    dict(item="min pairwise Hamming decrease", paper=pt["text_pairwise_hamming_decrease_range"][0], ours=claims.pairwise_decrease.min(), dec=3),
    dict(item="max pairwise Hamming decrease", paper=pt["text_pairwise_hamming_decrease_range"][1], ours=claims.pairwise_decrease.max(), dec=3),
    dict(item="min nearest-test decrease (stored)", paper=pt["text_nearest_test_decrease_range"][0], ours=claims.nearest_test_decrease_stored.min(), dec=3),
    dict(item="max nearest-test decrease (stored)", paper=pt["text_nearest_test_decrease_range"][1], ours=claims.nearest_test_decrease_stored.max(), dec=3),
    dict(item="Evo2 NLL improves in all 8 settings (1 = yes)", paper=1, ours=float(claims.evo2_improves.all()), atol=0),
    dict(item="pairwise decrease: recomputed vs stored (max abs diff)", paper=0.0, ours=float((claims.pairwise_decrease - claims.pairwise_decrease_stored).abs().max()), atol=1e-6),
]
show(cmp(rows))

status counts: {'match': 12}


,item,paper,ours,status,tol,tol_src,note
0,min fraction of pairs improved,0.930,9.300000e-01,match,0.005000,atol,
1,max fraction of pairs improved,0.980,9.800000e-01,match,0.005000,atol,
2,mean fraction improved (§3.1 'around 95%'),0.950,9.562500e-01,match,0.010000,atol,approximate claim ('around'): tolerance 0.01
3,pairs per setting (min over the 8),100.000,1.000000e+02,match,0.000000,atol,
4,pairs per setting (max over the 8),100.000,1.000000e+02,match,0.000000,atol,
5,all samples unique (1 = yes),1.000,1.000000e+00,match,0.000000,atol,
6,min pairwise Hamming decrease,0.008,7.608586e-03,match,0.000500,atol,
7,max pairwise Hamming decrease,0.020,1.967677e-02,match,0.000500,atol,
8,min nearest-test decrease (stored),0.003,3.150046e-03,match,0.000500,atol,
9,max nearest-test decrease (stored),0.012,1.240000e-02,match,0.000500,atol,


The Evo 2 held-out reference NLLs (100 random test windows per length: 1.3247 / 1.2820 / 1.2699 / 1.1855) are the same
for both rewards, as expected (same seed).

## RERUN = True — rerun on one GPU at small n

Every cell below reruns a real experiment of this section through the ported package `src/dmfm`
(the same `dmfm.experiments.*` entry points that `scripts/dna/*.sbatch` call for the full-size
runs), at a small number of samples/states, and prints the small-n result next to the stored
full-size number. They need a CUDA GPU and `source scripts/env.sh`; outputs go to
`outputs/dna/notebook_rerun/` (gitignored). Turn them on without editing the notebook with

```bash
DNA_RERUN=1 jupyter nbconvert --to notebook --execute --ExecutePreprocessor.kernel_name=multimfm \
    --output-dir <somewhere outside the repo> <this notebook>
```

(see `scripts` in the DNA provenance notes, or `python notebooks/_dna_helpers.py rerun <nb>`).
With `RERUN = False` each cell prints one line saying what it would rerun.

**Guided sampling is not bit-reproducible.** The value gradient is accumulated through a
nondeterministic CUDA backward and cuBLAS/cuDNN kernels depend on batch shape and GPU model, so
guided sequences and their scores differ between runs and from the stored runs; they agree
statistically. The *unguided* branch takes no gradient and is deterministic — it must match the
stored sequences exactly, and the cells below check that.

In [6]:
# Table 13 RERUN: exact motif-reward attainment at L=50 (dmfm.experiments.sample_reward_attainment,
# the flags of scripts/dna/table13_reward_attainment.sbatch): 8 pairs instead of 100, one of the
# eight (reward, length) settings. The dMFM flow map is the guidance posterior sampler.
if RERUN:
    from dmfm import paths as dpaths

    dev = require_gpu()
    N, REWARD, L = 8, "motif", 50
    rerun_banner(f"Table 13 - {REWARD} reward attainment at L={L}",
                 small=f"{N} guided/unguided pairs, target = 90th percentile of 1,024 base samples (unchanged)",
                 stored=f"100 pairs per setting, 8 settings (Slurm jobs 35383601-35383680)")
    print("The pair noise is seeded per batch start (seed + 100,000*L + batch_start) and the batch size\n"
          "is the original 8, so pairs 0-7 here are the stored run's first batch; the calibration that\n"
          "defines the target score keeps its original 1,024 samples, so target_score must reproduce.\n")
    d = rerun_dir("reward_attainment", REWARD)
    run_experiment("sample_reward_attainment", [
        "--lengths", L, "--reward", REWARD, "--dmfm_ckpt", f"{L}={dpaths.dmfm_ckpt(L)}",
        "--output_dir", d, "--seed", 20260731, "--n_samples", N, "--batch_size", 8,
        "--nfe_traj", 64, "--t_end", 0.999, "--guide_t_start", 0.50, "--guide_t_end", 0.95,
        "--mc", 8, "--mc_chunk", 4, "--guidance_frac", 8.0, "--grad_clip", 10.0, "--coeff_cap", 10.0,
        "--calibration_samples", 1024, "--nfe_calibration", 64, "--calibration_batch_size", 32,
        "--target_percentile", 0.90, "--reward_beta", 1.0, "--reward_scale", 1.0,
        "--motif", "TTTTTC", "--motif2", "AAAATT", "--motif_tau", 0.10, "--conjunction_tau", 0.10,
        "--bootstrap", 2000, "--device", dev])

    new = pd.read_csv(d / f"L{L}" / "reward_attainment_samples.csv")
    new_md = json.loads((d / f"L{L}" / "metadata.json").read_text())
    ref = load_csv(f"dna/exact_rewards/{REWARD}/L{L}/reward_attainment_samples.csv").set_index("sample_id")
    ref_md = load_json(f"dna/exact_rewards/{REWARD}/L{L}/metadata.json")
    ids = new.sample_id.tolist()
    nn, rr = new.set_index("sample_id").loc[ids], ref.loc[ids]
    display(stored_vs_small([
        dict(item="target score (90th pct of 1,024 base samples)", paper=ref_md["target_score"],
             ours=new_md["target_score"], atol=1e-4, note="same calibration seed and size"),
        dict(item="calibration score std", paper=ref_md["score_std"], ours=new_md["score_std"], atol=1e-4),
        dict(item=f"unguided sequences identical (of {len(ids)})", paper=len(ids), atol=0,
             ours=int((nn.sequence_unguided.to_numpy() == rr.sequence_unguided.to_numpy()).sum()),
             note="deterministic branch"),
        dict(item="max |unguided exact score - stored|", paper=0.0, atol=1e-5,
             ours=float(np.abs(nn.score_unguided.to_numpy() - rr.score_unguided.to_numpy()).max()),
             note="exact analytic reward on the decoded sequence"),
    ]))
    tgt = new_md["target_score"]
    delta = np.abs(nn.score_unguided.to_numpy() - tgt) - np.abs(nn.score_guided.to_numpy() - tgt)
    stored_row = settings[(settings.reward == REWARD) & (settings.L == L)].iloc[0]
    # A mean over 8 pairs cannot be expected inside the 100-pair CI, so the check is the other way round:
    # the stored full-size value against this run's own bootstrap CI (same estimator as the table above,
    # bootstrap_integers with the original seed + L).
    boot = lambda v: bootstrap_integers(v, 2000, 20260731 + L)
    display(agree_ci([
        dict(item=f"{REWARD} L{L} paired error reduction", stored=round(float(stored_row.paired_reduction), 4),
             ours=round(float(delta.mean()), 4), ci_ours=boot(delta),
             note=f"n={len(delta)} vs n={int(stored_row.n_pairs)}; the stored 100-pair CI is "
                  f"[{stored_row.ci_low:.3f}, {stored_row.ci_high:.3f}]"),
    ]))
    _f = float((delta > 0).mean())
    print(f"fraction of pairs improved: {_f:.3f} here (n={len(delta)}) vs {float(stored_row.frac_improved):.3f} in the "
          f"stored 100-pair run; 93-98% over the eight full settings. Not scored against a CI: when every pair "
          f"improves the bootstrap CI collapses to [1, 1], i.e. n=8 cannot resolve a 2% failure rate.")
    print(f"guided sequences identical to the stored run: "
          f"{int((nn.sequence_guided.to_numpy() == rr.sequence_guided.to_numpy()).sum())}/{len(ids)} "
          "(guided sampling is not bit-reproducible)")
    print("\nAll eight settings, full size: LENGTH=<50|100|200|400> REWARD=<motif|conjunction> "
          "sbatch scripts/dna/table13_reward_attainment.sbatch")
else:
    print("RERUN = False: Table 13 is rebuilt above from results/dna/exact_rewards/*/L*/reward_attainment_samples.csv.")
    print("RERUN = True regenerates 8 of the 100 motif/L=50 pairs with dmfm.experiments.sample_reward_attainment")
    print("and checks the target score and the unguided branch against the stored run.")

RERUN = False: Table 13 is rebuilt above from results/dna/exact_rewards/*/L*/reward_attainment_samples.csv.
RERUN = True regenerates 8 of the 100 motif/L=50 pairs with dmfm.experiments.sample_reward_attainment
and checks the target score and the unguided branch against the stored run.


In [7]:
# App. F.1 Evo 2-7B NLL of the generated sequences: PENDING as a rerun cell.
# dmfm.experiments.score_parent_reward_attainment_evo2 is ported, but Evo 2-7B needs its own
# environment (Python 3.12 / torch 2.7.1 / evo2 0.5.5, scripts/dna/evo2_environment.yml) and ~40 GB
# of GPU memory, which the notebook kernel (env multimfm) cannot provide; the Evo 2 numbers above
# come from the stored per-sequence CSVs. Run it out of band:
#
#   MODE=reward_attainment EVO2_ENV=<env> sbatch scripts/dna/score_evo2.sbatch
#     -> python -m dmfm.experiments.score_parent_reward_attainment_evo2 --run_dir <reward dir> ...
#
# (the Evo 2 environment and the regenerated Evo 2 results are the dna-regen agent's area;
#  notebook 09 covers them.)
_evo2 = "outputs/dna/notebook_rerun/reward_attainment/motif/L50" if RERUN else "results/dna/exact_rewards/motif/L50"
print("Evo 2 rerun: PENDING (separate environment; not run from this notebook).")
print("Stored Evo 2 NLL means used above (forward/RC-averaged, per set):")
display(load_csv("dna/exact_rewards/motif/L50/evo2/evo2_per_sequence.csv").groupby("set").evo2_nll.mean().round(4))
print("To score the sequences of the previous cell instead, point score_evo2.sbatch at", _evo2)

Evo 2 rerun: PENDING (separate environment; not run from this notebook).
Stored Evo 2 NLL means used above (forward/RC-averaged, per set):


set
guided          1.3436
heldout_test    1.3247
unguided        1.3591
Name: evo2_nll, dtype: float64

To score the sequences of the previous cell instead, point score_evo2.sbatch at results/dna/exact_rewards/motif/L50
